In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
# import from webdriver_manager (using underscore)
from webdriver_manager.chrome import ChromeDriverManager 
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as wait
from selenium.webdriver.support import expected_conditions as EC

##### Option 1: Open a browser normally

In [8]:
driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()))

# assign your website to scrape
# web = 'https://www.amazon.in'
web = "https://www.amazon.in/gp/bestsellers/home-improvement/ref=zg_bs_nav_home-improvement_0"
driver.get(web)
# keep this line of code at the bottom
driver.quit()

##### Option 2: Open a browser in the background

In [5]:
options = webdriver.ChromeOptions()
options.add_argument('--headless')
# create a driver object using driver_path as a parameter
driver = webdriver.Chrome(options = options, service = Service(ChromeDriverManager().install()))
# assign your website to scrape
# web = 'https://www.amazon.in'
web = "https://www.amazon.in/gp/bestsellers/home-improvement/ref=zg_bs_nav_home-improvement_0"
driver.get(web)
driver.quit()

In [68]:
from collections import namedtuple

In [69]:
help(namedtuple)

Help on function namedtuple in module collections:

namedtuple(typename, field_names, *, rename=False, defaults=None, module=None)
    Returns a new subclass of tuple with named fields.

    >>> Point = namedtuple('Point', ['x', 'y'])
    >>> Point.__doc__                   # docstring for the new class
    'Point(x, y)'
    >>> p = Point(11, y=22)             # instantiate with positional args or keywords
    >>> p[0] + p[1]                     # indexable like a plain tuple
    33
    >>> x, y = p                        # unpack like a regular tuple
    >>> x, y
    (11, 22)
    >>> p.x + p.y                       # fields also accessible by name
    33
    >>> d = p._asdict()                 # convert to a dictionary
    >>> d['x']
    11
    >>> Point(**d)                      # convert from a dictionary
    Point(x=11, y=22)
    >>> p._replace(x=100)               # _replace() is like str.replace() but targets named fields
    Point(x=100, y=22)



In [80]:
options = webdriver.ChromeOptions()
options.add_argument('--headless')
# create a driver object using driver_path as a parameter
driver = webdriver.Chrome(options = options, service = Service(ChromeDriverManager().install()))
# assign your website to scrape
web = "https://www.amazon.in/gp/bestsellers/home-improvement/ref=zg_bs_nav_home-improvement_0"
category = "home-improvement"
driver.get(web)
items = wait(driver, 10).until(EC.presence_of_all_elements_located((By.XPATH, '//div[contains(@id, "gridItemRoot")]')))

productDetails = namedtuple("Top100ProductDetails", ['asin', 'product_name', 'product_price', 'rating', 'number_of_ratings', 'rank', 'category', 'product_url', 'image_url', 'reviews_url'])
Top100ProductDetails = list()
for i,item in enumerate(items):
    rnk, name, num_ratings, price = item.find_element(By.XPATH, f'.//*[@id="p13n-asin-index-{i}"]/div').text.split("\n")
    asin = item.find_element(By.CLASS_NAME, 'p13n-sc-uncoverable-faceout').get_attribute("id")
    img_url = item.find_element(By.XPATH, f'//*[@id="{asin}"]/a/div/img').get_attribute("src")
    product_url = item.find_element(By.XPATH, f'//*[@id="{asin}"]/div/div/a').get_attribute("href")
    rating_obj = item.find_element(By.XPATH, f'//*[@id="{asin}"]/div/div/div[1]/div/a')
    rating = rating_obj.get_attribute("title")
    reviews_url = rating_obj.get_attribute("href")
    
    pd = productDetails(
        rank=rnk,
        product_name=name,
        number_of_ratings=num_ratings,
        product_price=price,
        asin=asin,
        image_url=img_url,
        product_url=product_url,
        category=category,
        rating=rating,
        reviews_url=reviews_url
    )
    Top100ProductDetails.append(pd)    

driver.quit()

In [91]:
Top100ProductDetails[0]._asdict()

{'asin': 'B0C7CYBGMX',
 'product_name': 'Hallstatt 2024 Upgraded Long Handle Microfiber Feather Ceiling Duster For Dust Cleaning Extendable Pole 30-100 Inch For Cleaning High Cobweb Stick High Ceiling Fan - Stainless Steel,Grey',
 'product_price': '₹269.00',
 'rating': '4.0 out of 5 stars',
 'number_of_ratings': '1,811',
 'rank': '#1',
 'category': 'home-improvement',
 'product_url': 'https://www.amazon.in/Microfiber-Feather-Ceiling-Duster-extendable/dp/B0C7CYBGMX/ref=zg_bs_g_home-improvement_d_sccl_1/260-6513830-3883811?psc=1',
 'image_url': 'https://images-eu.ssl-images-amazon.com/images/I/614BSL5MO9L._AC_UL600_SR600,400_.jpg',
 'reviews_url': 'https://www.amazon.in/product-reviews/B0C7CYBGMX/ref=zg_bs_g_home-improvement_d_sccl_1_cr/260-6513830-3883811'}

Testing Code

In [79]:
options = webdriver.ChromeOptions()
options.add_argument('--headless')
# create a driver object using driver_path as a parameter
driver = webdriver.Chrome(options = options, service = Service(ChromeDriverManager().install()))
# assign your website to scrape
web = "https://www.amazon.in/gp/bestsellers/home-improvement/ref=zg_bs_nav_home-improvement_0"
driver.get(web)
items = wait(driver, 10).until(EC.presence_of_all_elements_located((By.XPATH, '//div[contains(@id, "gridItemRoot")]')))
asins = list()
for i,item in enumerate(items):
    asin = item.find_element(By.CLASS_NAME, 'p13n-sc-uncoverable-faceout').get_attribute("id")
    # img_url = item.find_element(By.XPATH, f'//*[@id="{asin}"]/a/div/img').get_attribute("src")
    rating_obj = item.find_element(By.XPATH, f'//*[@id="{asin}"]/div/div/div[1]/div/a')
    rating = rating_obj.get_attribute("title")
    reviews_url = rating_obj.get_attribute("href")
    print(rating)
    asins.append(asin)
    # break
driver.quit()


4.0 out of 5 stars
4.3 out of 5 stars
4.2 out of 5 stars
4.2 out of 5 stars
4.2 out of 5 stars
4.2 out of 5 stars
4.5 out of 5 stars
4.4 out of 5 stars
4.0 out of 5 stars
4.5 out of 5 stars
4.0 out of 5 stars
4.6 out of 5 stars
4.4 out of 5 stars
4.2 out of 5 stars
4.4 out of 5 stars
3.5 out of 5 stars
4.3 out of 5 stars
4.0 out of 5 stars
4.4 out of 5 stars
4.0 out of 5 stars
3.6 out of 5 stars
3.9 out of 5 stars
4.4 out of 5 stars
4.5 out of 5 stars
3.8 out of 5 stars
3.9 out of 5 stars
4.2 out of 5 stars
4.1 out of 5 stars
4.2 out of 5 stars
4.6 out of 5 stars
